In [1]:
import numpy as np

# =========================================================
# SYSTEM SETUP & DATASETS
# =========================================================
print("========== EDIT DISTANCE & SPELLING CORRECTION SYSTEM ==========")

# Evaluation Dataset: A list of tuples containing (misspelled_word, correct_word)
evaluation_dataset = [
    ("speling", "spelling"),
    ("korrect", "correct"),
    ("wether", "weather"),
    ("recieve", "receive")
]

# Vocabulary with word frequencies for the Language Model P(w)
# In a real-world scenario, this would be extracted from a large text corpus.
vocab_freq = {
    "spelling": 200,
    "correct": 350,
    "weather": 250,
    "whether": 400,
    "receive": 150,
    "spilled": 50,
    "correction": 90,
    "korrect": 0  # Invalid word
}
total_words = sum(vocab_freq.values())

correct_predictions = 0

# =========================================================
# MINIMUM EDIT DISTANCE (MED) & NOISY CHANNEL MODEL
# =========================================================

# Process each misspelled word in the evaluation dataset
for typo, true_word in evaluation_dataset:
    print(f"\nAnalyzing misspelled word: '{typo}'")
    print(f"Target Correct Word: '{true_word}'")
    
    best_candidate = None
    best_score = -1.0
    best_distance = float('inf')
    
    # Iterate over all possible words in our vocabulary to find the best correction
    for candidate, freq in vocab_freq.items():
        
        # ---------------------------------------------------------
        # 1. Compute Minimum Edit Distance (Levenshtein Distance)
        # ---------------------------------------------------------
        m = len(typo)
        n = len(candidate)
        
        # Initialize a matrix to store the dynamic programming steps
        dp = np.zeros((m + 1, n + 1), dtype=int)
        
        for i in range(m + 1):
            dp[i][0] = i
        for j in range(n + 1):
            dp[0][j] = j
            
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                # If characters match, cost is 0. Otherwise, substitution cost is 1.
                if typo[i - 1] == candidate[j - 1]:
                    cost = 0
                else:
                    cost = 1
                
                dp[i][j] = min(
                    dp[i - 1][j] + 1,      # Deletion
                    dp[i][j - 1] + 1,      # Insertion
                    dp[i - 1][j - 1] + cost # Substitution
                )
        
        edit_distance = dp[m][n]
        
        # ---------------------------------------------------------
        # 2. Apply the Noisy Channel Model
        # Formula: P(w|x) is proportional to P(x|w) * P(w)
        # ---------------------------------------------------------
        
        # P(x|w): Error Model (Probability of typo given the intended word)
        # We simplify this by making it inversely proportional to the edit distance.
        # Adding 1 to avoid division by zero if the strings are identical.
        p_x_given_w = 1.0 / (edit_distance + 1)
        
        # P(w): Language Model (Prior probability of the candidate word)
        p_w = freq / total_words
        
        # Calculate final Noisy Channel score
        score = p_x_given_w * p_w
        
        # Update the most probable correction if this candidate scores higher
        if score > best_score:
            best_score = score
            best_candidate = candidate
            best_distance = edit_distance
            
    print(f"Most Probable Correction: '{best_candidate}' (Distance: {best_distance}, Score: {best_score:.6f})")
    
    # Check if the system's suggestion matches the true expected word
    if best_candidate == true_word:
        print("Result: Correctly fixed!")
        correct_predictions += 1
    else:
        print("Result: Failed to fix correctly.")

# =========================================================
# 3. SYSTEM PERFORMANCE EVALUATION
# =========================================================
print(f"\n========== SYSTEM EVALUATION ==========")
accuracy = (correct_predictions / len(evaluation_dataset)) * 100
print(f"Total Words Tested: {len(evaluation_dataset)}")
print(f"Correctly Corrected: {correct_predictions}")
print(f"System Accuracy: {accuracy:.2f}%")

========== EDIT DISTANCE & SPELLING CORRECTION SYSTEM ==========

Analyzing misspelled word: 'speling'
Target Correct Word: 'spelling'
Most Probable Correction: 'spelling' (Distance: 1, Score: 0.067114)
Result: Correctly fixed!

Analyzing misspelled word: 'korrect'
Target Correct Word: 'correct'
Most Probable Correction: 'correct' (Distance: 1, Score: 0.117450)
Result: Correctly fixed!

Analyzing misspelled word: 'wether'
Target Correct Word: 'weather'
Most Probable Correction: 'whether' (Distance: 1, Score: 0.134228)
Result: Failed to fix correctly.

Analyzing misspelled word: 'recieve'
Target Correct Word: 'receive'
Most Probable Correction: 'whether' (Distance: 6, Score: 0.038351)
Result: Failed to fix correctly.

========== SYSTEM EVALUATION ==========
Total Words Tested: 4
Correctly Corrected: 2
System Accuracy: 50.00%
